In [ ]:
!pip install earthengine-api geemap rasterio scikit-image

In [ ]:
import ee

ee.Authenticate()

ee.Initialize(project='YOUR_EE_PROJECT_ID') # Replace 'your-project-id' with your actual Google Cloud Project ID

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

BASE = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets"

folders = [
    "saudi_eastern_province/raw/dem/elevation",

    "saudi_eastern_province/processed/terrain",

    "saudi_eastern_province/grids/32x32",

    "saudi_eastern_province/grids/64x64"
]

for folder in folders:
    os.makedirs(os.path.join(BASE, folder), exist_ok=True)

print("DEM folders created.")

In [ ]:
saudi_roi = ee.Geometry.Rectangle([
    45.5,
    23.5,
    50.5,
    28.5
])

In [ ]:
dem = ee.Image("USGS/SRTMGL1_003")

In [ ]:
dem_saudi = dem.clip(saudi_roi)

In [ ]:
terrain = ee.Terrain.products(dem_saudi)

slope = terrain.select('slope')

In [ ]:
import geemap

Map = geemap.Map(center=[26.5, 48], zoom=6)

elevation_vis = {
    'min': 0,
    'max': 1000,
    'palette': ['blue', 'green', 'yellow', 'brown']
}

Map.addLayer(dem_saudi, elevation_vis, "Elevation")

Map

In [ ]:
Map = geemap.Map(center=[26.5, 48], zoom=6)

slope_vis = {
    'min': 0,
    'max': 30,
    'palette': ['white', 'yellow', 'orange', 'red']
}

Map.addLayer(slope, slope_vis, "Slope")

Map

In [ ]:
import geemap

output_tif = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/saudi_eastern_province/raw/dem/elevation/saudi_slope.tif"

geemap.ee_export_image(
    slope,
    filename=output_tif,
    scale=250,
    region=saudi_roi,
    file_per_band=False
)

print("Slope exported.")

In [ ]:
import rasterio

tif_path = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/saudi_eastern_province/raw/dem/elevation/saudi_slope.tif"

src = rasterio.open(tif_path)

slope_data = src.read(1)

print("Shape:", slope_data.shape)

In [ ]:
import numpy as np

print("Min:", np.min(slope_data))
print("Max:", np.max(slope_data))
print("Mean:", np.mean(slope_data))
print("NaNs:", np.isnan(slope_data).sum())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

plt.imshow(slope_data, cmap='terrain')

plt.title("Raw Saudi Terrain Slope")

plt.colorbar(label="Slope Degrees")

plt.show()

In [ ]:
slope_data = slope_data.astype(np.float32)

slope_data = np.nan_to_num(slope_data)

In [ ]:
def normalize(x):
    return (x - x.min()) / (x.max() - x.min())

slope_norm = normalize(slope_data)

In [ ]:
print("Min:", np.min(slope_norm))
print("Max:", np.max(slope_norm))

In [ ]:
!pip install scikit-image

In [ ]:
from skimage.transform import resize

In [ ]:
terrain_32 = resize(slope_norm, (32,32))

terrain_64 = resize(slope_norm, (64,64))

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(terrain_32, cmap='terrain')

plt.title("32x32 Terrain Grid")

plt.colorbar()

plt.show()

In [ ]:
GRID_BASE = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/saudi_eastern_province/grids"

In [ ]:
np.save(
    f"{GRID_BASE}/32x32/terrain.npy",
    terrain_32
)

np.save(
    f"{GRID_BASE}/64x64/terrain.npy",
    terrain_64
)

print("Terrain grids saved.")

In [ ]:
import os

print(os.listdir(f"{GRID_BASE}/32x32"))

In [ ]:
terrain_test = np.load(f"{GRID_BASE}/32x32/terrain.npy")

print(terrain_test.shape)

print(terrain_test.min())

print(terrain_test.max())

In [ ]:
print("Raw slope min:", slope_data.min())

print("Raw slope max:", slope_data.max())

print("Raw slope mean:", slope_data.mean())

In [ ]:
save_path = "/content/drive/MyDrive/PyroRL_Saudi_Project/visualizations/terrain_32.png"

plt.figure(figsize=(6,6))

plt.imshow(terrain_32, cmap='terrain')

plt.title("32x32 Terrain Grid")

plt.colorbar()

plt.savefig(save_path, dpi=300)

plt.show()

print("Visualization saved.")

In [ ]:
import json

metadata = {
    "dataset": "SRTM DEM",
    "region": "Saudi Eastern Province",
    "source_resolution_m": 250,
    "grid_sizes": [32, 64],
    "terrain_representation": "slope",
    "normalization": "minmax"
}

metadata_path = f"{GRID_BASE}/terrain_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata saved.")